# Semantic Search Pipeline: Chunking, Vectorization & Cosine Similarity

**Pipeline:** load text → fixed-size chunks with overlap → embed chunks → embed a query → cosine similarity → top 3 chunks

Upload your `.txt` file (at least 10 paragraphs) to the Colab **Files** panel, set its name below, then run the cells in order.

## Settings

In [ ]:
FILE_PATH = "sample_football_clubs.txt"    # your text file in the Colab Files panel (from the repo's data/ folder or your own)

CHUNK_UNIT = "characters"                  # "characters" or "tokens"
CHUNK_SIZE = 800                           # size of each chunk (in characters or tokens)
OVERLAP_PERCENT = 15                       # sliding window overlap, between 10 and 20 (%)

EMBEDDING_MODEL = "BAAI/bge-small-en-v1.5" # embedding model used for BOTH chunks and queries
TOP_K = 3                                  # maximum number of results to show
MIN_SCORE = 0.6                            # chunks below this similarity are not returned ("not in the document")

## Install

In [ ]:
!pip install -q sentence-transformers

## Step 1: Load the raw text file

In [ ]:
import os, re
import numpy as np
import pandas as pd
from google.colab import files

if not os.path.exists(FILE_PATH):
    print(f"{FILE_PATH} not found, please upload it:")
    FILE_PATH = list(files.upload().keys())[0]

with open(FILE_PATH, encoding="utf-8") as f:
    raw_text = f.read()

paragraphs = [" ".join(p.split()) for p in re.split(r"\n\s*\n", raw_text) if p.strip()]
if len(paragraphs) < 2:                                   # no blank lines: one paragraph per line
    paragraphs = [" ".join(p.split()) for p in raw_text.splitlines() if p.strip()]

text = " ".join(paragraphs)                               # clean text: one space between words

print(f"File: {FILE_PATH}")
print(f"Paragraphs: {len(paragraphs)} | Characters: {len(text):,} | Words: {len(text.split()):,}")
print("✅ At least 10 paragraphs" if len(paragraphs) >= 10 else "⚠️ The assignment needs at least 10 paragraphs")
print("\nStart of the text:\n", text[:300], "...")

## Step 2: Fixed-size window chunking with overlap
Each chunk is `CHUNK_SIZE` long. The next chunk starts `CHUNK_SIZE - overlap` later, so neighbouring chunks share `overlap` characters (or tokens) and no context is lost at the boundaries.

Example with 800 characters and 15% overlap: overlap = 120, so chunks start at 0, 680, 1360, ...

In [ ]:
assert 10 <= OVERLAP_PERCENT <= 20, "OVERLAP_PERCENT must be between 10 and 20"
overlap = round(CHUNK_SIZE * OVERLAP_PERCENT / 100)
step = CHUNK_SIZE - overlap

def sliding_windows(length, size, step):
    """Start and end positions of each window, sliding by `step`."""
    windows, start = [], 0
    while True:
        windows.append((start, min(start + size, length)))
        if start + size >= length:
            break
        start += step
    return windows

def chunk_text(text, unit, size, step):
    if unit == "characters":
        return [text[s:e] for s, e in sliding_windows(len(text), size, step)]
    if unit == "tokens":
        from transformers import AutoTokenizer
        tokenizer = AutoTokenizer.from_pretrained(EMBEDDING_MODEL)
        offsets = tokenizer(text, add_special_tokens=False, return_offsets_mapping=True)["offset_mapping"]
        # cut by token positions, then take the matching piece of the original text
        return [text[offsets[s][0]:offsets[e - 1][1]] for s, e in sliding_windows(len(offsets), size, step)]
    raise ValueError("CHUNK_UNIT must be 'characters' or 'tokens'")

chunks = chunk_text(text, CHUNK_UNIT, CHUNK_SIZE, step)

print(f"Chunk size: {CHUNK_SIZE} {CHUNK_UNIT} | Overlap: {overlap} {CHUNK_UNIT} ({OVERLAP_PERCENT}%) | Step: {step}")
print(f"Number of chunks: {len(chunks)}\n")

chunks_df = pd.DataFrame({"chunk_id": range(len(chunks)),
                          "characters": [len(c) for c in chunks],
                          "preview": [c[:80] + "..." for c in chunks]})
chunks_df

In [ ]:
# Check the overlap: the end of chunk 0 is repeated at the start of chunk 1
if len(chunks) > 1:
    shared = next(k for k in range(min(len(chunks[0]), len(chunks[1])), 0, -1) if chunks[0].endswith(chunks[1][:k]))
    print(f"End of chunk 0:   ...{chunks[0][-shared:]}")
    print(f"Start of chunk 1: {chunks[1][:shared]}...")
    print(f"\nShared text: {shared} characters")

## Step 3: Convert each chunk into a vector (embedding)

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(EMBEDDING_MODEL)
chunk_vectors = model.encode(chunks)                      # one vector per chunk

print(f"Model: {EMBEDDING_MODEL}")
print(f"Vectors: {chunk_vectors.shape[0]} chunks x {chunk_vectors.shape[1]} numbers each")
print("First 8 numbers of chunk 0:", np.round(chunk_vectors[0][:8], 4))

## Step 6 helper: Cosine similarity
cosine(A, B) = (A · B) / (‖A‖ × ‖B‖)

The result is between -1 and 1. The higher the score, the more similar the meaning.

In [ ]:
def cosine_similarity(query_vector, matrix):
    """Cosine similarity between one query vector and every row of a matrix."""
    dot_products = matrix @ query_vector                              # A · B for every chunk
    norms = np.linalg.norm(matrix, axis=1) * np.linalg.norm(query_vector)  # ‖A‖ × ‖B‖
    return dot_products / norms

## Steps 4–7: Ask a question → embed it → cosine similarity → top 3 chunks
Only chunks scoring at least `MIN_SCORE` are returned. If none do, the answer is reported as not being in the document.

In [ ]:
def semantic_search(question, top_k=TOP_K):
    query_vector = model.encode(question)                       # Step 5: same embedding model
    scores = cosine_similarity(query_vector, chunk_vectors)     # Step 6: cosine similarity
    ranking = np.argsort(scores)[::-1][:top_k]                  # Step 7: rank, highest first
    confident = [i for i in ranking if scores[i] >= MIN_SCORE]  # keep only confident matches

    print("=" * 80)
    print(f"Question: {question}")
    print("=" * 80)
    if not confident:
        print("❌ This is not in the document you provided.")
        print(f"   (the closest chunk scored {scores[ranking[0]]:.4f}, below the minimum of {MIN_SCORE})")
        return []

    print(f"Answers ({len(confident)} chunk(s) with similarity ≥ {MIN_SCORE}):\n")
    for rank, i in enumerate(confident, start=1):
        print(f"{rank}. Chunk {i} | Similarity score: {scores[i]:.4f}")
        print(f"   {chunks[i]}\n")
    return [(int(i), float(scores[i])) for i in confident]

question = input("Type your question about the text: ")      # Step 4: user query in plain text
results = semantic_search(question)

## Graph: similarity of every chunk to the question
Each bar is one chunk. The taller the bar, the more similar that chunk is to your question. Orange bars are the top 3 answers, and the dashed line is the average score.

In [ ]:
import matplotlib.pyplot as plt

def plot_similarity(question, top_k=TOP_K):
    scores = cosine_similarity(model.encode(question), chunk_vectors)
    top = np.argsort(scores)[::-1][:top_k]

    fig, ax = plt.subplots(figsize=(max(8, len(scores) * 0.45), 4.5))
    colors = ["tab:orange" if i in top else "tab:blue" for i in range(len(scores))]
    ax.bar(range(len(scores)), scores, color=colors)
    for rank, i in enumerate(top, start=1):
        ax.text(i, scores[i] + 0.005, f"#{rank}\n{scores[i]:.3f}", ha="center", va="bottom", fontsize=9, fontweight="bold")
    ax.axhline(scores.mean(), color="grey", ls="--", label=f"average score {scores.mean():.3f}")

    title = question if len(question) <= 70 else question[:67] + "..."
    ax.set(title=f'Similarity of every chunk to: "{title}"', xlabel="chunk number", ylabel="cosine similarity")
    ax.set_xticks(range(0, len(scores), max(1, len(scores) // 30)))
    ax.set_ylim(scores.min() - 0.05, scores.max() + 0.08)   # zoomed in so the differences are visible
    ax.legend(loc="lower right")
    plt.tight_layout()
    plt.show()

plot_similarity(question)

## Graph: similarity matrix
The top row compares the question with every chunk (the same scores as the search). The rest compares every chunk with every other chunk; the diagonal is always 1.

In [ ]:
all_vectors = np.vstack([model.encode(question), chunk_vectors])
names = ["Question"] + [f"Chunk {i}" for i in range(len(chunks))]
sim_matrix = np.array([cosine_similarity(v, all_vectors) for v in all_vectors])

fig, ax = plt.subplots(figsize=(max(7, len(names) * 0.6), max(6, len(names) * 0.55)))
im = ax.imshow(sim_matrix, cmap="viridis")
ax.set_xticks(range(len(names)), names, rotation=90)
ax.set_yticks(range(len(names)), names)
if len(names) <= 25:                                   # write the score in each square
    for r in range(len(names)):
        for c in range(len(names)):
            ax.text(c, r, f"{sim_matrix[r, c]:.2f}", ha="center", va="center", fontsize=7,
                    color="black" if sim_matrix[r, c] > sim_matrix.mean() else "white")
ax.axhline(0.5, color="red", lw=2)                     # red lines separate the question row/column
ax.axvline(0.5, color="red", lw=2)
ax.set_title(f'Cosine similarity matrix\n(top row = similarity to: "{question[:60]}")')
fig.colorbar(im, ax=ax, label="cosine similarity")
plt.tight_layout()
plt.show()

pd.DataFrame(sim_matrix.round(3), index=names, columns=names)

## Ask more questions
Type a question and press Enter. Leave it empty and press Enter to stop.

In [ ]:
while True:
    question = input("\nYour question (Enter to stop): ").strip()
    if not question:
        break
    semantic_search(question)
    plot_similarity(question)